In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from IPython.display import display, Markdown, Math, HTML
from optomechanics_levitation import input_beam_waist_from_filling_factor, electric_field_amplitude_from_power

In [3]:
filling_factor = 1.0
numerical_aperture = 0.75
focal_length = 1.0 # [mm]
refractive_index = 1.0

waist = input_beam_waist_from_filling_factor(filling_factor=filling_factor, numerical_aperture=numerical_aperture, focal_length_mm=focal_length, refractive_index=refractive_index)
display(Markdown(
    rf"""From a filling factor of $f_0 = {filling_factor:.2f}$, we obtain an entrance 
    beam waist of $w_0 = {waist:.3f}\,\mathrm{{mm}}$."""
))

From a filling factor of $f_0 = 1.00$, we obtain an entrance 
    beam waist of $w_0 = 0.750\,\mathrm{mm}$.

In [4]:
optical_power = 0.1 # [W]
impedance = 376.730313668 # [Ohms]
profile_power_integral = np.pi * (waist * 1e-3)**2 / 2 # Analitical solution for a guassian profile: in [m2]

amplitude_v_per_m = electric_field_amplitude_from_power(power_w=optical_power, profile_power_integral_m2=profile_power_integral, impedance_ohms=impedance)
display(Markdown(
    rf"""
Considering a Gaussian profile,

$$
u(x,y)=\exp\left(-\frac{{x^2+y^2}}{{w_0^2}}\right),
$$

with an optical power of ${optical_power:.1f}\,\mathrm{{W}}$, we obtain an electric field amplitude
$E_0 = {amplitude_v_per_m:.2f}\, \mathrm{{V/m}}$ 
"""
))


Considering a Gaussian profile,

$$
u(x,y)=\exp\left(-\frac{x^2+y^2}{w_0^2}\right),
$$

with an optical power of $0.1\,\mathrm{W}$, we obtain an electric field amplitude
$E_0 = 9234.41\, \mathrm{V/m}$ 


In [5]:
import os
import sys
import subprocess
import pandas as pd
from time import perf_counter
from threadpoolctl import threadpool_limits
from pyGDM2 import fields

from optomechanics_levitation.incident_field import evaluate_incident_field
from optomechanics_levitation.optical_fields.hermitegauss00 import hermite_gauss_00

In [6]:
def sampled_peak_memory(field_generator, **params):
    """Sample the PSS of the notebook kernel and its workers, in MiB."""
    monitor_code = """
import os
import sys
import select
import psutil

parent = psutil.Process(int(sys.argv[1]))
peak = 0
print("ready", flush=True)

while True:
    total = 0
    for process in [parent] + parent.children(recursive=True):
        if process.pid == os.getpid():
            continue
        try:
            total += process.memory_full_info().pss
        except psutil.NoSuchProcess:
            pass

    peak = max(peak, total)
    if select.select([sys.stdin], [], [], 0.005)[0]:
        break

print(peak / 1024**2, flush=True)
"""
    monitor = subprocess.Popen([sys.executable, "-c", monitor_code, str(os.getpid())], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    monitor.stdout.readline()
    try:
        result = evaluate_incident_field(field_generator, **params)
    finally:
        output, _ = monitor.communicate(input="\n")
    return float(output.strip())

In [7]:
# Keep the observation positions and quadrature fixed.
points_per_axis = 65
quadrature_points = 24
cpu_counts = [1, 2, 4, 6, 8, 16]
repeats = 3

axis_nm = np.linspace(-2000, 2000, points_per_axis)
x_nm, y_nm = np.meshgrid(axis_nm, axis_nm, indexing="ij")
positions = np.column_stack((x_nm.ravel(), y_nm.ravel(), np.zeros(x_nm.size)))

# Preserve your current physical beam parameters.
hermitegauss_params = dict(
    positions_nm=positions,
    wavelength_nm=1550.0,
    environment={"eps1": 1.0, "eps2": 1.0, "eps3": 1.0},
    amplitude_v_per_m=amplitude_v_per_m,
    field="both",
    theta=0, NA=numerical_aperture, f=focal_length, w0=waist,
    xSpot=0, ySpot=0, zSpot=0, kSign=-1, phase=0
)

In [8]:
configurations = [("Original (PyGDM)", fields.HermiteGauss00, 1)]
configurations += [("Vectorized CPU", hermite_gauss_00, n_cpu) for n_cpu in cpu_counts]
rows = []
# Limit internal BLAS pools; N_cpu controls the field generator's workers.
with threadpool_limits(limits=1):
    for mode, generator, n_cpu in configurations:
        params = dict(hermitegauss_params, N_cpu=n_cpu)
        if mode == "Vectorized CPU":
            params["quadrature_points"] = quadrature_points

        # Warm up on a small subset, outside the timed region.
        evaluate_incident_field(generator, **dict(params, positions_nm=positions[:10]))

        times = []
        for _ in range(repeats):
            start = perf_counter()
            E, H = evaluate_incident_field(generator, **params)
            times.append(perf_counter() - start)

        elapsed_time = float(np.median(times))
        if mode == "Original (PyGDM)":
            original_time = elapsed_time
            E_original, H_original = E, H
            E_difference = H_difference = np.nan
        else:
            E_difference = np.linalg.norm(E - E_original) / np.linalg.norm(E_original)
            H_difference = np.linalg.norm(H - H_original) / np.linalg.norm(H_original)

        del E, H

        # Measure memory separately so monitoring does not affect timing.
        peak_memory_mib = sampled_peak_memory(generator, **params)
        rows.append({
            "Mode": mode,
            "Workers": n_cpu,
            "Time (s)": elapsed_time,
            "Speedup": original_time / elapsed_time,
            "E relative difference": E_difference,
            "H relative difference": H_difference,
            "Sampled peak PSS (MiB)": peak_memory_mib
        })

incident_field_benchmarks = pd.DataFrame(rows)

In [9]:
def sci_notation(value, decimals):
    """ Function to write values in scientific notation """
    mantissa, exponent = f"{value:.{decimals}e}".split("e")
    return rf"{mantissa} \times 10^{{{int(exponent)}}}" 

benchmark_table = """
| Mode | Workers | Time [s] | Speedup | $\\mathbf{E}$ relative difference | $\\mathbf{H}$ relative difference | Sampled peak PSS [MiB] |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
"""

for row in rows:
    E_text = "—" if row["Mode"] == "Original (PyGDM)" else rf"${sci_notation(row['E relative difference'], 2)}$"
    H_text = "—" if row["Mode"] == "Original (PyGDM)" else rf"${sci_notation(row['H relative difference'], 2)}$"
    benchmark_table += f"| **{row['Mode']}** | {row['Workers']} | {row['Time (s)']:.4f} | ${row['Speedup']:.2f}\\times$ | {E_text} | {H_text} | {row['Sampled peak PSS (MiB)']:.1f} |\n"

display(Markdown(f"**Positions:** {len(positions):,} · **Optimized quadrature nodes:** {quadrature_points} · **Timing repetitions:** {repeats}"))
display(Markdown(benchmark_table))

**Positions:** 4,225 · **Optimized quadrature nodes:** 24 · **Timing repetitions:** 3


| Mode | Workers | Time [s] | Speedup | $\mathbf{E}$ relative difference | $\mathbf{H}$ relative difference | Sampled peak PSS [MiB] |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| **Original (PyGDM)** | 1 | 6.8864 | $1.00\times$ | — | — | 320.2 |
| **Vectorized CPU** | 1 | 0.0133 | $517.01\times$ | $2.78 \times 10^{-8}$ | $2.44 \times 10^{-8}$ | 226.1 |
| **Vectorized CPU** | 2 | 0.0157 | $438.37\times$ | $2.78 \times 10^{-8}$ | $2.44 \times 10^{-8}$ | 234.5 |
| **Vectorized CPU** | 4 | 0.0108 | $636.18\times$ | $2.78 \times 10^{-8}$ | $2.44 \times 10^{-8}$ | 238.8 |
| **Vectorized CPU** | 6 | 0.0118 | $581.97\times$ | $2.78 \times 10^{-8}$ | $2.44 \times 10^{-8}$ | 242.5 |
| **Vectorized CPU** | 8 | 0.0149 | $461.09\times$ | $2.78 \times 10^{-8}$ | $2.44 \times 10^{-8}$ | 245.4 |
| **Vectorized CPU** | 16 | 0.0229 | $300.33\times$ | $2.78 \times 10^{-8}$ | $2.44 \times 10^{-8}$ | 252.1 |
